# 2025-03-30-词袋模型-阿里云

## 题目内容

假设你团队正在开发一个文本分类模型，用于将客户评论分类为正面或负面。由于文本数据具有高维度的特性，模型训练和预测的效率受到影响。你提议使用卡方检验进行特征选择，挑选出与分类任务最相关的词汇，降低数据的维度，从而提高模型的性能。

请你编写一个程序，使用卡方检验对给定的文本数据集进行特征选择。

具体要求如下：
1. 读取输入数据集，包含多篇标注了类别的文本文档。
2. 提取特征，采用词频($Bag-of-Words$)模型，将文本转换为特征向量。(不能忽视单词字母大小写)
3. 计算每个特征(词)的卡方统计量，衡量其与类别标签的相关性。
4. 根据卡方统计量选择前 $(k)$ 个最重要的特征。
5. 输出选定的特征列表。

## 输入描述

- 第一行包含一个整数 $(N)$，表示文档的数量。
- 接下来的 $(N)$ 行，每行包含一个文档，格式为: `< label >\t < text >`，文档的类别，取值为 "positive" 或 "negative"。
- `<\t>` 一个制表符，分隔类别标签和文档内容。
- `< text >`: 文档的内容，由若干单词组成，单词之间用空格分隔。
- 最后一行包含一个整数 $(k)$，表示需要选择的特征数量。

## 输出描述

输出 $(k)$ 行，每行包含一个单词(特征)，按照卡方统计量从大到小排序。如果多个特征的卡方值相同，按字母顺序升序排列。

## 补充说明

- 卡方检验公式
  对于每个特征(单词)，卡方统计量计算公式为:
  $$
  x^2 = \sum_{i=1}^2 \sum_{j=1}^2 \frac{(O_{ij} - E_{ij})^2}{E_{ij}}
  $$
  其中:
  - $(O_{ij})$ 是观察到的频数，表示特征是否出现和类别的四种组合情况，
  - $E_{ij}$ 是期望频数，按照独立性假设计算:
    $$E_{ij} = \frac{(特征是否出现的行数) \times (类别的列和)}{总样本数}$$

## 样例

样例1
```
输入:
6
positive I love this movie
negative I hate this movie
positive This film was fantastic
negative This film was terrible
positive What a great experience
negative What a bad experience
3

输出:
bad
fantastic
great
```

说明
- 步骤1：读取 6 篇文档及其类别标签。
- 步骤2：统计每个单词在不同类别中的出现次数，计算卡方统计量。
- 步骤3：根据卡方值从大到小排序，选择前3个特征。自测验入
- 步骤4：输出选定的特征列表。

样例2
```
输入:
2
1 1

输出:
-1
```

In [12]:
# 处理输入
n = int(input())
docs = []
for _ in range(n):
    docs.append(input())
m = int(input())
docs

['positive\tgood I love love this poor terrible this terrible great',
 'positive\tawesome average I awesome good fantastic boring love hate',
 'negative\tgood bad great hate good',
 'negative\tmovie terrible boring',
 'positive\tterrible film love bad terrible fantastic dull good awesome poor',
 'positive\twonderful fantastic I experience dull poor boring',
 'negative\tgreat movie average poor',
 'negative\tterrible bad good love dull bad love',
 'positive\texperience awesome average',
 'negative\twonderful love good boring movie boring good',
 'negative\taverage this bad film wonderful amazing',
 'negative\tdull experience film hate love great',
 'negative\tfilm I I',
 'positive\twas was great movie awesome',
 'positive\tlove fantastic film film experience dull I',
 'negative\taverage was boring average wonderful love terrible film wonderful boring',
 'positive\tdull fantastic this terrible amazing',
 'positive\tpoor bad wonderful this amazing wonderful love was poor',
 'positive\tter

In [27]:
# 处理数据
positive_all = 0
negative_all = 0
word_dict = {}
for doc in docs:
    if '\t' in doc:
        label, text = doc.split('\t')
        text = text.split()
    else:
        temp = doc.split()
        label = temp[0]
        text = temp[1:]
    if label == "positive":
        positive_all += 1
    else:
        negative_all += 1
    words = set(text)
    
    for word in words:
        if word not in word_dict:
            word_dict[word] = [0, 0] # 初始正负样本count       
        if label == "positive":
            word_dict[word][1] += 1
        else:
            word_dict[word][0] += 1 
word_dict    

{'great': [4, 2],
 'I': [1, 4],
 'love': [5, 6],
 'this': [2, 5],
 'poor': [1, 4],
 'terrible': [4, 5],
 'good': [3, 5],
 'awesome': [0, 5],
 'average': [3, 2],
 'fantastic': [0, 6],
 'boring': [4, 2],
 'hate': [2, 1],
 'bad': [4, 4],
 'movie': [3, 3],
 'film': [4, 2],
 'dull': [2, 5],
 'experience': [1, 6],
 'wonderful': [3, 3],
 'amazing': [1, 2],
 'was': [2, 2]}

In [28]:
'''
计算相关统计量

# TP: 特征出现在positive
# FP: 特征没有出现在positive
# TN: 特征出现在negative
# FN: 特征没有出现在negative
# FP = positive_all - TP
# FN = negative_all - TN
'''
features_x2 = []
for word, counts in word_dict.items():
    x_2 = 0.0
    TP = counts[1]
    TN = counts[0]
    FP = positive_all - counts[1]
    FN = negative_all - counts[0]
    T = TP + TN
    F = FP + FN
    if T > 0 and positive_all > 0: # word 出现在positive
        E = (T * positive_all)/n
        x_2 += (TP - E) ** 2/E
    if T > 0 and negative_all > 0: # word 没有出现在positive
        E = (T * negative_all)/n
        x_2 += (TN - E) ** 2/E
    if F > 0 and positive_all > 0: # word 出现在negative
        E = (F * positive_all)/n
        x_2 += (FP - E) ** 2/E
    if F > 0 and negative_all > 0:  # word 没有出现在negative
        E = (F * negative_all)/n
        x_2 += (FN - E) ** 2/E
    features_x2.append((word, x_2))
features_x2.sort(key=lambda item: [-item[1], item[0]])
features_x2   

[('fantastic', 6.244343891402714),
 ('awesome', 4.914529914529915),
 ('experience', 3.4894917582417584),
 ('boring', 1.7761689291101055),
 ('film', 1.7761689291101055),
 ('great', 1.7761689291101055),
 ('I', 1.4330769230769231),
 ('poor', 1.4330769230769231),
 ('dull', 0.9098901098901101),
 ('this', 0.9098901098901101),
 ('hate', 0.7548717948717947),
 ('average', 0.7096581196581198),
 ('bad', 0.21230769230769228),
 ('good', 0.17839743589743595),
 ('amazing', 0.14448717948717954),
 ('movie', 0.14049773755656114),
 ('wonderful', 0.14049773755656114),
 ('was', 0.08380566801619424),
 ('love', 0.03350815850815853),
 ('terrible', 0.005616605616605646)]

## 完整代码

In [ ]:
import sys
def feature_exaction():
    lines = sys.stdin.read().splitlines()
    if not lines:
        print(-1)
        return
    try:
        # 第一行为文档数量N
        n = int(lines[0])
    except:
        print(-1)
        return

    # 检查是否有足够的行：N行文档 + 1 行k
    if len(lines) != n + 2:
        print(-1)
        return
    try:
        m = int(lines[-1])
    except:
        print(-1)
        return
    
    # 处理数据
    positive_all = 0
    negative_all = 0
    word_dict = {}
    for i in range(1, n+1):
        doc = lines[i]
        if '\t' in doc:
            label, text = doc.split('\t')
            text = text.split()
        else:
            temp = doc.split()
            label = temp[0]
            text = temp[1:]
        if label == "positive":
            positive_all += 1
        else:
            negative_all += 1
        words = set(text)
        
        for word in words:
            if word not in word_dict:
                word_dict[word] = [0, 0] # 初始正负样本count       
            if label == "positive":
                word_dict[word][1] += 1
            else:
                word_dict[word][0] += 1 
    features_x2 = []
    for word, counts in word_dict.items():
        x_2 = 0.0
        TP = counts[1]
        TN = counts[0]
        FP = positive_all - counts[1]
        FN = negative_all - counts[0]
        T = TP + TN
        F = FP + FN
        if T > 0 and positive_all > 0: # word 出现在positive
            E = (T * positive_all)/n
            x_2 += (TP - E) ** 2/E
        if T > 0 and negative_all > 0: # word 没有出现在positive
            E = (T * negative_all)/n
            x_2 += (TN - E) ** 2/E
        if F > 0 and positive_all > 0: # word 出现在negative
            E = (F * positive_all)/n
            x_2 += (FP - E) ** 2/E
        if F > 0 and negative_all > 0:  # word 没有出现在negative
            E = (F * negative_all)/n
            x_2 += (FN - E) ** 2/E
        features_x2.append((word, x_2))
    features_x2.sort(key=lambda item: [-item[1], item[0]])
    for i in range(min(m, len(features_x2))):
        print(features_x2[i][0])

if __name__ == "__main__":
    feature_exaction()